<a href="https://colab.research.google.com/github/ionikim/epilepsy_pediatrics_EEG/blob/main/src/01_loading/preprocessing_210326_interictal_changed_weighted.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In this file, we
1. automate the data preprocessing and
2. create multiplex horizontal visibility graphs for a certain time window in a file (edge list + adjacency matrix)

Comments:
- What is updated from the last file, we calculated weights between edges.
- Top 5% strongest threshold is applied.
- Check the nodes and edges file and import in the neo4j.
- To create visualization with graphs which are created from this notebook, check the file "src/04_visualization/Graphs_to_Gepx.ipynb"

In [ ]:
!pip install ts2vg
!pip install mne
%cd /content
!git clone https://github.com/ionikim/epilepsy_pediatrics_EEG.git
%cd /content/epilepsy_pediatrics_EEG
from pathlib import Path
import mne
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import glob
import json
from ts2vg import HorizontalVG
import scipy.sparse as sp
from scipy.sparse import save_npz

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 40.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 56.1 MB/s eta 0:00:00
/content
Cloning into 'epilepsy_pediatrics_EEG'...
remote: Enumerating objects: 490, done.
remote: Counting objects: 100% (172/172), done.
remote: Compressing objects: 100% (112/112), done.
remote: Total 490 (delta 100), reused 92 (delta 59), pack-reused 318 (from 2)
Receiving objects: 100% (490/490), 69.66 MiB | 18.43 MiB/s, done.
Resolving deltas: 100% (200/200), done.
/content/epilepsy_pediatrics_EEG


In [ ]:
RAW_DIR = Path("data/raw")

PROCESSED_DIR = Path("data/processed")
LABELED_DIR   = PROCESSED_DIR / "labeled_signals"
FILEMETA_DIR  = PROCESSED_DIR / "file_metadata"

WINDOWS_DIR   = Path("data/windows")
WINDOWSIG_DIR = WINDOWS_DIR / "signals"
WINDOWMETA_DIR = WINDOWS_DIR / "metadata"

for d in [LABELED_DIR, FILEMETA_DIR, WINDOWSIG_DIR, WINDOWMETA_DIR]:
    d.mkdir(parents=True, exist_ok=True)

In [ ]:
print("CWD:", Path.cwd())
print("RAW_DIR exists:", Path("data/raw").exists())
print("Files in raw:")
for p in Path("data/raw").glob("*"):
    print(p.name)

CWD: /content/epilepsy_pediatrics_EEG
RAW_DIR exists: True
Files in raw:
chb01_03.edf.seizures
chb01_03.edf
.gitattributes
chb01_10.edf
.gitkeep


In [ ]:
def save_json(data, out_path):
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)

In [ ]:
def get_seizure_period(annotation_file):
    with open(annotation_file, "rb") as f:
        byte_array = f.read()

    start = int(bin(byte_array[38])[2:] + bin(byte_array[41])[2:], 2)
    length = byte_array[49]
    return start, length

In [ ]:
def process_edf_with_labels(edf_file, seizure_file=None):
    edf_file = Path(edf_file)
    seizure_file = Path(seizure_file) if seizure_file is not None else None

    raw = mne.io.read_raw_edf(str(edf_file), preload=True, verbose=False)
    raw.filter(1., 45.)
    raw.set_eeg_reference('average')

    electrode_names = raw.ch_names
    sfreq = raw.info["sfreq"]
    n_samples = len(raw.times)
    time_marks = np.arange(n_samples) / sfreq
    electrode_data = raw.get_data()

    df = pd.DataFrame(electrode_data.T, columns=electrode_names)
    df.index = time_marks
    df.index.name = "Time (s)"

    seizure_start = None
    seizure_length = None
    seizure_end = None

    if seizure_file is not None:
        seizure_start, seizure_length = get_seizure_period(seizure_file)
        seizure_end = seizure_start + seizure_length

        df["label"] = df.index.map(
            lambda t: "ictal" if seizure_start <= t <= seizure_end else "interictal"
        )
        has_seizure = True
    else:
        df["label"] = "interictal"
        has_seizure = False

    file_meta = {
        "edf_file": str(edf_file),
        "seizure_file": str(seizure_file) if seizure_file else None,
        "sampling_rate_hz": float(sfreq),
        "channel_names": list(electrode_names),
        "n_channels": int(len(electrode_names)),
        "n_samples": int(n_samples),
        "duration_s": float(n_samples / sfreq),
        "has_seizure": bool(has_seizure),
        "seizure_start_s": float(seizure_start) if seizure_start is not None else None,
        "seizure_length_s": float(seizure_length) if seizure_length is not None else None,
        "seizure_end_s": float(seizure_end) if seizure_end is not None else None,
    }

    return df, file_meta

In [ ]:
def extract_window(df, window_size_s=10, mode="ictal", strategy="first"):
    sfreq = int(round(1 / (df.index[1] - df.index[0])))
    window_size_samples = window_size_s * sfreq

    if mode == "ictal":
        indices = np.where(df["label"] == "ictal")[0]
    elif mode == "interictal":
        indices = np.where(df["label"] == "interictal")[0]
    else:
        raise ValueError("mode must be 'ictal' or 'interictal'")

    if len(indices) == 0:
        raise ValueError(f"No {mode} samples found.")

    valid_starts = [
        idx for idx in indices
        if idx + window_size_samples <= len(df)
        and (df["label"].iloc[idx: idx + window_size_samples] == mode).all()
    ]

    if not valid_starts:
        raise ValueError(f"No contiguous {mode} block of {window_size_s}s found.")

    if mode == "ictal":
        onset_idx = int(indices[0])
        offset_idx = int(indices[-1])

        if strategy == "first":
            start_idx = int(valid_starts[0])
        elif strategy == "middle":
            center_idx = (onset_idx + offset_idx) // 2
            start_idx = center_idx - (window_size_samples // 2)
        elif strategy == "last":
            start_idx = offset_idx - window_size_samples + 1
        else:
            raise ValueError("For ictal, strategy must be 'first', 'middle', or 'last'")

        if start_idx < 0 or start_idx + window_size_samples > len(df):
            raise ValueError(f"Invalid ictal window for strategy='{strategy}'")

        if not (df["label"].iloc[start_idx: start_idx + window_size_samples] == "ictal").all():
            raise ValueError(f"Ictal window for strategy='{strategy}' is not fully ictal.")

    else:  # interictal
        if strategy == "first":
            start_idx = int(valid_starts[0])
        elif strategy == "middle":
            start_idx = int(valid_starts[len(valid_starts) // 2])
        elif strategy == "last":
            start_idx = int(valid_starts[-1])
        else:
            raise ValueError("For interictal, strategy must be 'first', 'middle', or 'last'")

    window = df.iloc[start_idx: start_idx + window_size_samples]
    return window, start_idx

In [ ]:
def make_signal_id(edf_path):
    return Path(edf_path).stem

In [ ]:
def process_ictal_interictal_pair(
    ictal_edf,
    ictal_seizure_file,
    interictal_edf,
    window_size_s=10,
    ictal_strategy="middle",
    interictal_strategy="middle"
):
    ictal_edf = Path(ictal_edf)
    ictal_seizure_file = Path(ictal_seizure_file)
    interictal_edf = Path(interictal_edf)

    # -------------------------
    # 1) ictal source 처리
    # -------------------------
    ictal_df, ictal_meta = process_edf_with_labels(ictal_edf, ictal_seizure_file)
    ictal_id = make_signal_id(ictal_edf)

    ictal_labeled_out = LABELED_DIR / f"{ictal_id}.parquet"
    ictal_df.to_parquet(ictal_labeled_out)

    ictal_meta_out = FILEMETA_DIR / f"{ictal_id}.json"
    save_json(ictal_meta, ictal_meta_out)

    ictal_window, ictal_start_idx = extract_window(
        ictal_df,
        window_size_s=window_size_s,
        mode="ictal",
        strategy=ictal_strategy
    )

    ictal_window_id = f"window_ictal_{ictal_id}"
    ictal_window_out = WINDOWSIG_DIR / f"{ictal_window_id}.parquet"
    ictal_window.to_parquet(ictal_window_out)

    # -------------------------
    # 2) interictal source
    # -------------------------
    interictal_df, interictal_meta = process_edf_with_labels(interictal_edf, seizure_file=None)
    interictal_id = make_signal_id(interictal_edf)

    interictal_labeled_out = LABELED_DIR / f"{interictal_id}.parquet"
    interictal_df.to_parquet(interictal_labeled_out)

    interictal_meta_out = FILEMETA_DIR / f"{interictal_id}.json"
    save_json(interictal_meta, interictal_meta_out)

    interictal_window, interictal_start_idx = extract_window(
        interictal_df,
        window_size_s=window_size_s,
        mode="interictal",
        strategy=interictal_strategy
    )

    interictal_window_id = f"window_interictal_{interictal_id}"
    interictal_window_out = WINDOWSIG_DIR / f"{interictal_window_id}.parquet"
    interictal_window.to_parquet(interictal_window_out)

    # -------------------------
    # 3) window metadata 저장
    # -------------------------
    ictal_sfreq = int(round(1 / (ictal_df.index[1] - ictal_df.index[0])))
    interictal_sfreq = int(round(1 / (interictal_df.index[1] - interictal_df.index[0])))

    windows_index = pd.DataFrame([
        {
            "window_id": ictal_window_id,
            "source_edf": str(ictal_edf),
            "label": "ictal",
            "selection_rule": ictal_strategy,
            "start_idx": ictal_start_idx,
            "end_idx": ictal_start_idx + len(ictal_window) - 1,
            "start_time_s": float(ictal_window.index[0]),
            "end_time_s": float(ictal_window.index[-1]),
            "n_samples": int(len(ictal_window)),
            "window_size_s": int(window_size_s),
            "sampling_rate_hz": float(ictal_sfreq),
        },
        {
            "window_id": interictal_window_id,
            "source_edf": str(interictal_edf),
            "label": "interictal",
            "selection_rule": interictal_strategy,
            "start_idx": interictal_start_idx,
            "end_idx": interictal_start_idx + len(interictal_window) - 1,
            "start_time_s": float(interictal_window.index[0]),
            "end_time_s": float(interictal_window.index[-1]),
            "n_samples": int(len(interictal_window)),
            "window_size_s": int(window_size_s),
            "sampling_rate_hz": float(interictal_sfreq),
        }
    ])

    windows_index_out = WINDOWMETA_DIR / "windows_index.csv"
    windows_index.to_csv(windows_index_out, index=False)

    # -------------------------
    # 4) processing log save
    # -------------------------
    processing_log = pd.DataFrame([
        {
            "source_type": "ictal",
            "edf_file": str(ictal_edf),
            "annotation_file": str(ictal_seizure_file),
            "status": "success",
            "message": "ictal window extracted successfully",
            "saved_labeled_df": str(ictal_labeled_out),
            "saved_window": str(ictal_window_out),
            "saved_metadata": str(ictal_meta_out),
        },
        {
            "source_type": "interictal",
            "edf_file": str(interictal_edf),
            "annotation_file": None,
            "status": "success",
            "message": "interictal window extracted successfully",
            "saved_labeled_df": str(interictal_labeled_out),
            "saved_window": str(interictal_window_out),
            "saved_metadata": str(interictal_meta_out),
        }
    ])

    processing_log_out = PROCESSED_DIR / "processing_log.csv"
    processing_log.to_csv(processing_log_out, index=False)

    # -------------------------
    # 5) print
    # -------------------------
    print("\n=== Ictal source ===")
    print("EDF file               :", ictal_edf)
    print("Seizure annotation     :", ictal_seizure_file)
    print("Saved labeled signal   :", ictal_labeled_out)
    print("Saved file metadata    :", ictal_meta_out)
    print("Saved window           :", ictal_window_out)
    print("Window ID              :", ictal_window_id)
    print("Start time (s)         :", float(ictal_window.index[0]))
    print("End time (s)           :", float(ictal_window.index[-1]))

    print("\n=== Interictal source ===")
    print("EDF file               :", interictal_edf)
    print("Saved labeled signal   :", interictal_labeled_out)
    print("Saved file metadata    :", interictal_meta_out)
    print("Saved window           :", interictal_window_out)
    print("Window ID              :", interictal_window_id)
    print("Strategy               :", interictal_strategy)
    print("Start time (s)         :", float(interictal_window.index[0]))
    print("End time (s)           :", float(interictal_window.index[-1]))

    print("\nSaved window index     :", windows_index_out)
    print("Saved processing log   :", processing_log_out)

    return {
        "ictal_df": ictal_df,
        "interictal_df": interictal_df,
        "ictal_window": ictal_window,
        "interictal_window": interictal_window,
        "windows_index": windows_index,
        "processing_log": processing_log,
    }

In [ ]:
result = process_ictal_interictal_pair(
    ictal_edf=RAW_DIR / "chb01_03.edf",
    ictal_seizure_file=RAW_DIR / "chb01_03.edf.seizures",
    interictal_edf=RAW_DIR / "chb01_10.edf",
    window_size_s=10,
    ictal_strategy="middle",
    interictal_strategy="middle"
)

/tmp/ipykernel_18885/609150053.py:5: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(str(edf_file), preload=True, verbose=False)


Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 1 - 45 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 1.00
- Lower transition bandwidth: 1.00 Hz (-6 dB cutoff frequency: 0.50 Hz)
- Upper passband edge: 45.00 Hz
- Upper transition bandwidth: 11.25 Hz (-6 dB cutoff frequency: 50.62 Hz)
- Filter length: 845 samples (3.301 s)

EEG channel type selected for re-referencing
Applying average reference.
Applying a custom ('EEG',) reference.


/tmp/ipykernel_18885/609150053.py:5: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(str(edf_file), preload=True, verbose=False)


Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 1 - 45 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 1.00
- Lower transition bandwidth: 1.00 Hz (-6 dB cutoff frequency: 0.50 Hz)
- Upper passband edge: 45.00 Hz
- Upper transition bandwidth: 11.25 Hz (-6 dB cutoff frequency: 50.62 Hz)
- Filter length: 845 samples (3.301 s)

EEG channel type selected for re-referencing
Applying average reference.
Applying a custom ('EEG',) reference.

=== Ictal source ===
EDF file               : data/raw/chb01_03.edf
Seizure annotation     : data/raw/chb01_03.edf.seizures
Saved labeled signal   : data/processed/labeled_signals/chb01_03.parquet
Saved file metadata    : data/processed/file_metadata/chb01_03.json
Saved window           : data/windows/signals/window_ictal_chb

In [ ]:
def make_multiplex_node_table(window_df, graph_label):
    """
    Each node = one electrode at one time point.
    """
    electrode_cols = [c for c in window_df.columns if c != "label"]
    n_timepoints = len(window_df)

    rows = []
    for electrode in electrode_cols:
        for t in range(n_timepoints):
            node_name = f"{electrode}_t{t}"
            rows.append({
                "Id": node_name,
                "Label": node_name,
                "electrode": electrode,
                "time_idx": int(t),
                "layer": electrode,
                "graph_label": graph_label
            })

    return pd.DataFrame(rows)

In [ ]:
def prune_edges_by_quantile(edge_rows, keep_ratio=0.05):
    """
    edge_rows: list of dicts, each dict must contain 'weight'
    keep_ratio=0.05 means keep top 5% highest-weight edges
    """
    if edge_rows is None or len(edge_rows) == 0:
        return []

    if keep_ratio is None:
        return edge_rows

    if not (0 < keep_ratio <= 1):
        raise ValueError("keep_ratio must be in (0, 1].")

    edge_df = pd.DataFrame(edge_rows)

    if "weight" not in edge_df.columns:
        raise ValueError("'weight' column not found in edge_rows.")

    threshold = edge_df["weight"].quantile(1 - keep_ratio)

    pruned_df = edge_df[edge_df["weight"] >= threshold].copy()

    return pruned_df.to_dict("records")

In [ ]:
def build_multiplex_hvg(window_df, intra_keep_ratio=None):

    electrode_cols = [c for c in window_df.columns if c != "label"]

    if len(electrode_cols) == 0:
        raise ValueError("No electrode columns found. 'window_df' seems empty or only has 'label'.")

    n_electrodes = len(electrode_cols)
    n_timepoints = len(window_df)
    n_nodes_total = n_electrodes * n_timepoints

    print(f"Electrodes        : {n_electrodes}")
    print(f"Time points       : {n_timepoints}")
    print(f"Total nodes       : {n_nodes_total}")

    def node_id(electrode, t):
        return f"{electrode}_t{t}"

    node_labels = [
        node_id(electrode, t)
        for electrode in electrode_cols
        for t in range(n_timepoints)
    ]

    label_to_idx = {label: idx for idx, label in enumerate(node_labels)}

    node_index = {
        (electrode, t): node_id(electrode, t)
        for electrode in electrode_cols
        for t in range(n_timepoints)
    }

    adj_sparse = sp.lil_matrix((n_nodes_total, n_nodes_total), dtype=np.float32)

    intra_rows = []
    inter_rows = []
    layer_info = {}

    # --------------------------------------------------
    # 1) Intra-layer HVG edges
    # --------------------------------------------------
    for layer_idx, electrode in enumerate(electrode_cols):
        ts = window_df[electrode].to_numpy()

        hvg = HorizontalVG()
        hvg.build(ts)

        layer_intra_rows = []

        for t_i, t_j in hvg.edges:
            t_i = int(t_i)
            t_j = int(t_j)

            u = node_id(electrode, t_i)
            v = node_id(electrode, t_j)

            weight = abs(float(ts[t_i]) - float(ts[t_j]))

            layer_intra_rows.append({
                "Source": u,
                "Target": v,
                "source_label": u,
                "target_label": v,
                "edge_type": "intra",
                "layer": electrode,
                "weight": weight
            })

        n_before = len(layer_intra_rows)

        if intra_keep_ratio is not None:
            layer_intra_rows = prune_edges_by_quantile(
                layer_intra_rows,
                keep_ratio=intra_keep_ratio
            )

        n_after = len(layer_intra_rows)

        for row in layer_intra_rows:
            intra_rows.append(row)

            i = label_to_idx[row["Source"]]
            j = label_to_idx[row["Target"]]
            adj_sparse[i, j] = row["weight"]
            adj_sparse[j, i] = row["weight"]

        layer_info[electrode] = {
            "layer_index": int(layer_idx),
            "n_intra_edges_before": int(n_before),
            "n_intra_edges_after": int(n_after)
        }

    print(f"Intra-layer edges : {len(intra_rows)}")

    # --------------------------------------------------
    # 2) Inter-layer edges
    # --------------------------------------------------
    for t in range(n_timepoints):
        for i in range(n_electrodes):
            for j in range(i + 1, n_electrodes):
                e_i = electrode_cols[i]
                e_j = electrode_cols[j]

                u = node_id(e_i, t)
                v = node_id(e_j, t)

                inter_rows.append({
                    "Source": u,
                    "Target": v,
                    "source_label": u,
                    "target_label": v,
                    "edge_type": "inter",
                    "layer": f"{e_i}<->{e_j}",
                    "weight": 0.1
                })

                ui = label_to_idx[u]
                vj = label_to_idx[v]
                adj_sparse[ui, vj] = 1.0
                adj_sparse[vj, ui] = 1.0

    print(f"Inter-layer edges : {len(inter_rows)}")

    edge_list = pd.DataFrame(intra_rows + inter_rows)
    print(f"Total edges       : {len(edge_list)}")

    return edge_list, adj_sparse.tocsr(), node_index, layer_info, node_labels

In [ ]:
def save_multiplex_hvg_outputs(
    edge_list,
    adj_sparse,
    node_index,
    node_labels,
    layer_info,
    window_df,
    graph_id,
    source_edf,
    window_id,
    label,
    intra_keep_ratio=None,
    out_root="data/graphs"
):
    out_root = Path(out_root)
    out_root.mkdir(parents=True, exist_ok=True)

    edges_dir = out_root / "edges"
    nodes_dir = out_root / "nodes"
    adj_dir = out_root / "adjacency_sparse"
    meta_dir = out_root / "metadata"

    for d in [edges_dir, nodes_dir, adj_dir, meta_dir]:
        d.mkdir(parents=True, exist_ok=True)

    # 1) node table
    node_df = make_multiplex_node_table(window_df, graph_label=label)

    # 2) edge list
    edge_path = edges_dir / f"{graph_id}_edges.csv"
    edge_list.to_csv(edge_path, index=False)

    # 3) node list
    node_path = nodes_dir / f"{graph_id}_nodes.csv"
    node_df.to_csv(node_path, index=False)

    # 4) sparse adjacency
    adj_path = adj_dir / f"{graph_id}_adjacency_sparse.npz"
    save_npz(adj_path, adj_sparse)

    # 5) node labels
    node_labels_path = meta_dir / f"{graph_id}_node_labels.json"
    with open(node_labels_path, "w", encoding="utf-8") as f:
        json.dump(node_labels, f, indent=2, ensure_ascii=False)

    # 6) layer info
    layer_info_df = (
        pd.DataFrame(layer_info)
        .T
        .reset_index()
        .rename(columns={"index": "electrode"})
    )
    layer_info_path = meta_dir / f"{graph_id}_layer_info.csv"
    layer_info_df.to_csv(layer_info_path, index=False)

    # 7) node index
    node_index_str = {
        f"{k[0]}__t{k[1]}": v
        for k, v in node_index.items()
    }
    node_index_path = meta_dir / f"{graph_id}_node_index.json"
    with open(node_index_path, "w", encoding="utf-8") as f:
        json.dump(node_index_str, f, indent=2, ensure_ascii=False)

    # 8) metadata
    metadata = {
        "graph_id": graph_id,
        "source_edf": str(source_edf),
        "window_id": window_id,
        "label": label,
        "n_nodes": int(node_df.shape[0]),
        "n_edges": int(edge_list.shape[0]),
        "n_intra_edges": int((edge_list["edge_type"] == "intra").sum()),
        "n_inter_edges": int((edge_list["edge_type"] == "inter").sum()),
        "n_layers": int(node_df["electrode"].nunique()),
        "n_timepoints": int(len(window_df)),
        "adjacency_format": "scipy_sparse_npz",
        "intra_edge_weight": "abs(amplitude_i - amplitude_j)",
        "intra_keep_ratio": intra_keep_ratio
    }

    metadata_path = meta_dir / f"{graph_id}_metadata.json"
    with open(metadata_path, "w", encoding="utf-8") as f:
        json.dump(metadata, f, indent=2, ensure_ascii=False)

    return {
        "edges": str(edge_path),
        "nodes": str(node_path),
        "adjacency_sparse": str(adj_path),
        "node_labels": str(node_labels_path),
        "layer_info": str(layer_info_path),
        "node_index": str(node_index_path),
        "metadata": str(metadata_path),
    }

In [ ]:
# -------------------------
# ICTAL window
# -------------------------
ictal_window = result["ictal_window"]
ictal_window_id = "window_ictal_chb01_03"
ictal_source_edf = RAW_DIR / "chb01_03.edf"
ictal_graph_id = "graph_ictal_chb01_03"
intra_keep_ratio = 0.05

display(ictal_window.head())

ictal_edge_list, ictal_adj_sparse, ictal_node_index, ictal_layer_info, ictal_node_labels = build_multiplex_hvg(
    ictal_window,
    intra_keep_ratio=intra_keep_ratio
)

saved_paths_ictal = save_multiplex_hvg_outputs(
    edge_list=ictal_edge_list,
    adj_sparse=ictal_adj_sparse,
    node_index=ictal_node_index,
    node_labels=ictal_node_labels,
    layer_info=ictal_layer_info,
    window_df=ictal_window,
    graph_id=ictal_graph_id,
    source_edf=ictal_source_edf,
    window_id=ictal_window_id,
    label="ictal",
    intra_keep_ratio=intra_keep_ratio,
    out_root="data/graphs"
)

# -------------------------
# INTERICTAL window
# -------------------------
interictal_window = result["interictal_window"]
interictal_window_id = "window_interictal_chb01_10"
interictal_source_edf = RAW_DIR / "chb01_10.edf"
interictal_graph_id = "graph_interictal_chb01_10"

display(interictal_window.head())

interictal_edge_list, interictal_adj_sparse, interictal_node_index, interictal_layer_info, interictal_node_labels = build_multiplex_hvg(
    interictal_window,
    intra_keep_ratio=intra_keep_ratio
)

saved_paths_interictal = save_multiplex_hvg_outputs(
    edge_list=interictal_edge_list,
    adj_sparse=interictal_adj_sparse,
    node_index=interictal_node_index,
    node_labels=interictal_node_labels,
    layer_info=interictal_layer_info,
    window_df=interictal_window,
    graph_id=interictal_graph_id,
    source_edf=interictal_source_edf,
    window_id=interictal_window_id,
    label="interictal",
    intra_keep_ratio=intra_keep_ratio,
    out_root="data/graphs"
)

print("ICTAL FILES:")
print(saved_paths_ictal)

print("\nINTERICTAL FILES:")
print(saved_paths_interictal)

,FP1-F7,F7-T7,T7-P7,P7-O1,FP1-F3,F3-C3,C3-P3,P3-O1,FP2-F4,F4-C4,...,T8-P8-0,P8-O2,FZ-CZ,CZ-PZ,P7-T7,T7-FT9,FT9-FT10,FT10-T8,T8-P8-1,label
Time (s),,,,,,,,,,,,,,,,,,,,,
3011.000000,0.000132,0.000031,0.000009,-0.000041,0.000155,0.000027,-0.000032,-0.000020,0.000029,0.000008,...,0.000070,-0.000070,-0.000004,-0.000034,-0.000039,-0.000037,-0.000044,-0.000052,0.000070,ictal
3011.003906,0.000126,0.000036,0.000004,-0.000040,0.000168,0.000008,-0.000026,-0.000023,0.000086,-0.000017,...,0.000043,-0.000077,-0.000031,-0.000033,-0.000036,-0.000042,-0.000024,-0.000040,0.000043,ictal
3011.007812,0.000125,0.000032,-0.000007,-0.000041,0.000164,0.000001,-0.000027,-0.000029,0.000183,-0.000061,...,-0.000031,-0.000078,-0.000067,-0.000037,-0.000031,-0.000044,0.000010,-0.000018,-0.000031,ictal
3011.011719,0.000120,0.000026,-0.000015,-0.000039,0.000137,0.000011,-0.000028,-0.000029,0.000252,-0.000101,...,-0.000107,-0.000074,-0.000098,-0.000041,-0.000021,-0.000041,0.000053,0.000007,-0.000107,ictal
3011.015625,0.000104,0.000024,-0.000012,-0.000031,0.000104,0.000024,-0.000022,-0.000020,0.000242,-0.000115,...,-0.000141,-0.000067,-0.000112,-0.000040,-0.000006,-0.000033,0.000090,0.000029,-0.000141,ictal


Electrodes        : 23
Time points       : 2560
Total nodes       : 58880
Intra-layer edges : 5845
Inter-layer edges : 647680
Total edges       : 653525


,FP1-F7,F7-T7,T7-P7,P7-O1,FP1-F3,F3-C3,C3-P3,P3-O1,FP2-F4,F4-C4,...,T8-P8-0,P8-O2,FZ-CZ,CZ-PZ,P7-T7,T7-FT9,FT9-FT10,FT10-T8,T8-P8-1,label
Time (s),,,,,,,,,,,,,,,,,,,,,
1795.000000,0.000029,0.000012,-0.000018,0.000011,0.000052,0.000003,-0.000027,0.000006,0.000012,-0.000014,...,0.000001,0.000028,0.000006,-0.000033,-0.000004,-0.000014,-0.000016,-2.354649e-05,0.000001,interictal
1795.003906,0.000024,0.000014,-0.000012,0.000012,0.000042,0.000010,-0.000026,0.000012,0.000005,-0.000011,...,-0.000001,0.000017,0.000012,-0.000029,-0.000006,-0.000016,-0.000011,-1.578393e-05,-0.000001,interictal
1795.007812,0.000008,0.000019,-0.000007,0.000014,0.000014,0.000025,-0.000019,0.000013,-0.000011,-0.000001,...,-0.000001,0.000006,0.000020,-0.000020,-0.000003,-0.000017,-0.000008,-5.837100e-06,-0.000001,interictal
1795.011719,-0.000014,0.000023,-0.000008,0.000015,-0.000019,0.000038,-0.000010,0.000008,-0.000026,0.000011,...,0.000003,0.000004,0.000024,-0.000011,0.000008,-0.000015,-0.000006,2.991133e-07,0.000003,interictal
1795.015625,-0.000035,0.000021,-0.000010,0.000018,-0.000044,0.000041,-0.000007,0.000003,-0.000037,0.000020,...,0.000010,0.000009,0.000023,-0.000005,0.000019,-0.000008,-0.000003,7.686013e-07,0.000010,interictal


Electrodes        : 23
Time points       : 2560
Total nodes       : 58880
Intra-layer edges : 5841
Inter-layer edges : 647680
Total edges       : 653521
ICTAL FILES:
{'edges': 'data/graphs/edges/graph_ictal_chb01_03_edges.csv', 'nodes': 'data/graphs/nodes/graph_ictal_chb01_03_nodes.csv', 'adjacency_sparse': 'data/graphs/adjacency_sparse/graph_ictal_chb01_03_adjacency_sparse.npz', 'node_labels': 'data/graphs/metadata/graph_ictal_chb01_03_node_labels.json', 'layer_info': 'data/graphs/metadata/graph_ictal_chb01_03_layer_info.csv', 'node_index': 'data/graphs/metadata/graph_ictal_chb01_03_node_index.json', 'metadata': 'data/graphs/metadata/graph_ictal_chb01_03_metadata.json'}

INTERICTAL FILES:
{'edges': 'data/graphs/edges/graph_interictal_chb01_10_edges.csv', 'nodes': 'data/graphs/nodes/graph_interictal_chb01_10_nodes.csv', 'adjacency_sparse': 'data/graphs/adjacency_sparse/graph_interictal_chb01_10_adjacency_sparse.npz', 'node_labels': 'data/graphs/metadata/graph_interictal_chb01_10_node_l

In [ ]:
print("ICTAL source EDF       :", ictal_source_edf)
print("ICTAL window id        :", ictal_window_id)
print("ICTAL graph id         :", ictal_graph_id)

print("INTERICTAL source EDF  :", interictal_source_edf)
print("INTERICTAL window id   :", interictal_window_id)
print("INTERICTAL graph id    :", interictal_graph_id)

ICTAL source EDF       : data/raw/chb01_03.edf
ICTAL window id        : window_ictal_chb01_03
ICTAL graph id         : graph_ictal_chb01_03
INTERICTAL source EDF  : data/raw/chb01_10.edf
INTERICTAL window id   : window_interictal_chb01_10
INTERICTAL graph id    : graph_interictal_chb01_10


In [35]:
import pandas as pd

# ICTAL
nodes_ictal = pd.read_csv("data/graphs/nodes/graph_ictal_chb01_03_nodes.csv")
edges_ictal = pd.read_csv("data/graphs/edges/graph_ictal_chb01_03_edges.csv")

print("ICTAL NODES COLUMNS:")
print(nodes_ictal.columns.tolist())

print("\nICTAL EDGES COLUMNS:")
print(edges_ictal.columns.tolist())

display(nodes_ictal.head())
display(edges_ictal.head())

ICTAL NODES COLUMNS:
['Id', 'Label', 'electrode', 'time_idx', 'layer', 'graph_label']

ICTAL EDGES COLUMNS:
['Source', 'Target', 'source_label', 'target_label', 'edge_type', 'layer', 'weight']


,Id,Label,electrode,time_idx,layer,graph_label
0,FP1-F7_t0,FP1-F7_t0,FP1-F7,0,FP1-F7,ictal
1,FP1-F7_t1,FP1-F7_t1,FP1-F7,1,FP1-F7,ictal
2,FP1-F7_t2,FP1-F7_t2,FP1-F7,2,FP1-F7,ictal
3,FP1-F7_t3,FP1-F7_t3,FP1-F7,3,FP1-F7,ictal
4,FP1-F7_t4,FP1-F7_t4,FP1-F7,4,FP1-F7,ictal


,Source,Target,source_label,target_label,edge_type,layer,weight
0,FP1-F7_t1923,FP1-F7_t1924,FP1-F7_t1923,FP1-F7_t1924,intra,FP1-F7,0.000062
1,FP1-F7_t1669,FP1-F7_t1670,FP1-F7_t1669,FP1-F7_t1670,intra,FP1-F7,0.000079
2,FP1-F7_t1665,FP1-F7_t1670,FP1-F7_t1665,FP1-F7_t1670,intra,FP1-F7,0.000066
3,FP1-F7_t1418,FP1-F7_t1670,FP1-F7_t1418,FP1-F7_t1670,intra,FP1-F7,0.000065
4,FP1-F7_t1410,FP1-F7_t1411,FP1-F7_t1410,FP1-F7_t1411,intra,FP1-F7,0.000072
